# BRCA driver evidence audit
This notebook checks the two linked evidence tables and their pinned source snapshots. The 42 DriverMONI positives are used as the paper-attributed CGC v95 subset because the full export is unavailable. OncoKB gene summaries and IntOGen statistical calls remain candidates until context review; no training labels are changed. The literature assertions are a limited reviewed spot-check, and source/cohort counts do not establish independence.


## Setup


In [1]:
from pathlib import Path
from collections import Counter
import contextlib, csv, io, json, sys, tempfile
from unittest.mock import patch
ROOT = Path.cwd()
while not (ROOT / "configs/brca_driver_evidence.json").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Open this notebook from the reproduction repository")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.preprocessing.acquisition import FileCache, open_text_source
from src.preprocessing.brca_driver_evidence import run_driver_evidence, validate_tables, zip_table
from src.preprocessing.common import sha256_file
from src.preprocessing.brca_sources import read_labels
DATA = ROOT / "data/brca_driver_evidence"
manifest = json.loads((DATA / "manifest.json").read_text())
cache = FileCache(ROOT / "data/cache", offline=True, lock=manifest)
paths = {row["role"]: cache.get(row["role"], row["specification"]) for row in manifest["sources"]}
def csv_rows(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))
evidence = csv_rows(DATA / "brca_driver_evidence.csv")
statuses = csv_rows(DATA / "brca_gene_status.csv")
summary = json.loads((DATA / "summary.json").read_text())
print("Cached source snapshots:", len(paths))


Cached source snapshots: 11


## Input and output integrity


In [2]:
for artifact in manifest["outputs"]:
    assert sha256_file(DATA / artifact["file"]) == artifact["sha256"]
for artifact in manifest["inputs"]:
    assert sha256_file(artifact["path"]) == artifact["sha256"]
assert sha256_file(ROOT / "src/preprocessing/brca_driver_evidence.py") == manifest["implementation"]["code_sha256"]
original_manifest = json.loads((ROOT / "data/brca/manifest.json").read_text())
for artifact in original_manifest["outputs"]:
    assert sha256_file(ROOT / "data/brca" / artifact["file"]) == artifact["sha256"]
print("All evidence outputs, local inputs, source snapshots and 11 existing BRCA artifacts match their recorded hashes.")
print("Source coverage:", json.dumps(summary["coverage"], indent=2))
print("IntOGen version labels:", json.dumps(summary["versions"]["IntOGen"], indent=2))


All evidence outputs, local inputs, source snapshots and 11 existing BRCA artifacts match their recorded hashes.
Source coverage: {
  "CGC": "42 DriverMONI positives used as paper-attributed v95 subset at user request; full export unavailable, extraction and tiers unverified",
  "OncoKB": "Full public gene-list and somatic/germline curated-gene exports; tumour/alteration-level export requires authorization and was not acquired",
  "IntOGen": "Complete downloaded compendium, unfiltered assertions and cohort metadata; BRCA records selected by exact cancer code",
  "literature": "Literature spot-check from the BRCA label audit; not an exhaustive review or a CGC reconstruction."
}
IntOGen version labels: {
  "intogen_drivers": {
    "archive_directory": "2024-06-18_IntOGen-Drivers",
    "readme_release": "20230531",
    "advertised_release": "2024-09-20"
  },
  "intogen_cohorts": {
    "archive_directory": "2024-06-18_IntOGen-Cohorts",
    "readme_release": "not_declared",
    "advertised_

## Links and biological settings


In [3]:
integer_columns = ["source_record_count", "reviewed_somatic_study_count", "intogen_brca_pass_cohorts",
    "intogen_primary_untreated_pass_cohorts", "intogen_same_project_or_possible_overlap_cohorts",
    "oncokb_breast_summary_mentions", "cgc_v95_subset_member", "in_original_features",
    "in_benchmark_evaluation", "in_canonical_features", "review_required"]
for row in statuses:
    for key in integer_columns:
        row[key] = int(row[key])
    row["benchmark_label"] = int(row["benchmark_label"]) if row["benchmark_label"] else ""
labels = set(read_labels(paths["benchmark_labels"]))
validate_tables(evidence, statuses, labels)
cgc = [row for row in evidence if row["source_resource"] == "CGC"]
assert len(cgc) == 42 and {row["gene_ens"] for row in cgc} == labels
assert all(row["setting"] == "not_specified" and not row["source_tier"] and row["eligible_for_somatic_status"] == "0" for row in cgc)
supported = [row for row in statuses if row["evidence_status"] == "supported_somatic_BRCA_driver"]
assert len(supported) == 7 and all(row["benchmark_label"] == 0 for row in supported)
assert all(row["negative_label_status"] in ("supplied_positive", "unlabelled_not_confirmed_negative") for row in statuses)
assert sum(row["benchmark_label"] == 1 for row in statuses) == 42
assert sum(not row["gene_ens"] for row in statuses) == 9
print("Evidence rows:", len(evidence), "Gene/status entities:", len(statuses))
print("Status counts:", dict(Counter(row["evidence_status"] for row in statuses)))
print("Reviewed somatic support with original benchmark zero:", sorted(row["gene_symbol"] for row in supported))
print("All 42 original positives preserved; nine unresolved source entities are visible; no confirmed negatives assigned.")


Evidence rows: 3787 Gene/status entities: 16980
Status counts: {'insufficient_evidence': 14903, 'unresolved_candidate': 2070, 'supported_somatic_BRCA_driver': 7}
Reviewed somatic support with original benchmark zero: ['CBFB', 'KMT2C', 'NF1', 'PIK3R1', 'PTEN', 'RUNX1', 'SF3B1']
All 42 original positives preserved; nine unresolved source entities are visible; no confirmed negatives assigned.


## Independent source-row reconciliation


In [4]:
raw, _ = zip_table(paths["intogen_drivers"], "Unfiltered_drivers.tsv", ("SYMBOL", "COHORT", "CANCER_TYPE"))
compendium, _ = zip_table(paths["intogen_drivers"], "Compendium_Cancer_Genes.tsv", ("SYMBOL", "COHORT", "CANCER_TYPE"))
cohorts, _ = zip_table(paths["intogen_cohorts"], "cohorts.tsv", ("COHORT", "CANCER"))
cohorts = {row["COHORT"]: row for row in cohorts}
raw = [row for row in raw if row["CANCER_TYPE"] == "BRCA"]
intogen = [row for row in evidence if row["source_resource"] == "IntOGen"]
serial = lambda row: json.dumps(row, sort_keys=True)
assert Counter(serial(row) for row in raw) == Counter(serial(json.loads(row["source_payload_json"])["unfiltered"]) for row in intogen)
accepted = [row for row in raw if row["DRIVER"] == "True" and row["FILTER"] == "PASS"]
compendium = [row for row in compendium if row["CANCER_TYPE"] == "BRCA"]
key = lambda row: (row["SYMBOL"], row["TRANSCRIPT"], row["COHORT"])
assert {key(row) for row in accepted} == {key(row) for row in compendium}
assert len(raw) == 1338 and len(accepted) == 287 and len({row["SYMBOL"] for row in accepted}) == 118
for row in intogen:
    payload = json.loads(row["source_payload_json"])
    assert payload["cohort"] == cohorts[row["cohort_id"]]
    passing = payload["unfiltered"]["DRIVER"] == "True" and payload["unfiltered"]["FILTER"] == "PASS"
    assert (row["assertion"] == "BRCA_somatic_driver_call") == passing
for role, assertion in [("oncokb_gene_list", "broad_cancer_gene_list_membership"), ("oncokb_curated_genes", "curated_gene_context")]:
    with paths[role].open(encoding="utf-8-sig", newline="") as handle:
        source_rows = list(csv.DictReader(handle, delimiter="\t"))
    output_rows = [row for row in evidence if row["source_resource"] == "OncoKB" and row["assertion"] == assertion]
    assert Counter(serial(row) for row in source_rows) == Counter(serial(json.loads(row["source_payload_json"])) for row in output_rows)
    assert all(row["eligible_for_somatic_status"] == "0" for row in output_rows)
    print(role, "exact source records:", len(source_rows))
print("IntOGen: 1,338 exact BRCA records, 287 accepted calls, 118 genes; cohort fields and rejection flags reconcile.")


oncokb_gene_list exact source records: 1247
oncokb_curated_genes exact source records: 1144
IntOGen: 1,338 exact BRCA records, 287 accepted calls, 118 genes; cohort fields and rejection flags reconcile.


## Locked offline replay


In [5]:
inputs = {row["role"]: Path(row["path"]) for row in manifest["inputs"]}
with tempfile.TemporaryDirectory(prefix="brca-evidence-replay-") as directory:
    with patch.object(FileCache, "_download", side_effect=AssertionError("Offline replay attempted network")), contextlib.redirect_stdout(io.StringIO()):
        replay_summary = run_driver_evidence(manifest["configuration"],
            cache=FileCache(ROOT / "data/cache", offline=True, lock=manifest), output_dir=directory,
            canonical_path=inputs["canonical_features"], reviewed_path=inputs["reviewed_assertions"], lock=manifest)
    replay_manifest = json.loads((Path(directory) / "manifest.json").read_text())
    assert replay_manifest["outputs"] == manifest["outputs"]
    assert replay_summary == summary
print("Locked offline replay reproduces both tables and summary byte-for-byte, with network acquisition disabled.")


Locked offline replay reproduces both tables and summary byte-for-byte, with network acquisition disabled.
